# GeoWatch — Phase 5: Automated Intelligence Report

This notebook demonstrates Milestone 5, the final piece of the original
MVP scope: tying stored events and imagery-based wildfire detection into
a single exportable intelligence report.

Two data sources feed into a report:

1. **Stored events** (e.g. from NASA FIRMS ingestion, Milestone 3) —
   real-world observations with a location and timestamp.
2. **A specific wildfire detection result** (Milestone 2's dNBR-based
   burned-area/severity analysis), optionally attached when a pre/post-fire
   imagery comparison was run for the same AOI/period.

The report's `limitations` section is generated from what data actually
went into it — not fixed boilerplate. A report built from FIRMS events
only reads differently from one that also includes a severity analysis.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from datetime import datetime, timedelta, timezone

import numpy as np

from src.geospatial.aoi import AOI
from src.detection.wildfire import detect_wildfire
from src.remote_sensing.nbr import compute_nbr, compute_dnbr
from src.reporting.reports import generate_intelligence_report
from src.types import ConfidenceScore, EventStatus, EventType, EvidenceLevel, GeoWatchEvent


## 1. Simulate stored events

In a real run these would come from `PostgresEventStore.find_events_within_bbox()`
after a live FIRMS fetch (see notebook 04). Here we construct a small,
realistic set directly so this notebook runs standalone without requiring
a database or a FIRMS_MAP_KEY.

In [2]:
aoi = AOI(label="Southern Africa (demo AOI)", west=10.0, south=-35.0, east=40.0, north=-10.0)

base_time = datetime(2026, 8, 24, tzinfo=timezone.utc)

events = [
    GeoWatchEvent(
        event_id=f"firms:demo:{i}",
        event_type=EventType.WILDFIRE,
        latitude=-18.5 + i * 0.1,
        longitude=16.2 + i * 0.05,
        detected_at=base_time + timedelta(hours=i),
        observation_time=base_time + timedelta(hours=i),
        source="NASA FIRMS (VIIRS_NOAA20_NRT)",
        evidence_level=EvidenceLevel.OBSERVED,
        confidence=ConfidenceScore(
            value=[0.3, 0.6, 0.9][i % 3], basis="demo", evidence_level=EvidenceLevel.OBSERVED
        ),
        status=EventStatus.NEW,
    )
    for i in range(6)
]

print(f"{len(events)} simulated FIRMS detections over the demo AOI.")


6 simulated FIRMS detections over the demo AOI.


## 2. Run a wildfire detection for the same AOI

Reusing the synthetic pre/post-fire scene approach from notebook 03 —
same honest labeling: this is a constructed scene, not a real location's
imagery.

In [3]:
SIZE = 50
nir_pre = np.clip(0.50 + np.random.default_rng(1).normal(0, 0.01, (SIZE, SIZE)), 0, 1)
swir_pre = np.clip(0.15 + np.random.default_rng(2).normal(0, 0.01, (SIZE, SIZE)), 0, 1)
nir_post = nir_pre.copy()
swir_post = swir_pre.copy()
nir_post[:, SIZE//2:] -= 0.35
swir_post[:, SIZE//2:] += 0.15
nir_post = np.clip(nir_post, 0, 1)
swir_post = np.clip(swir_post, 0, 1)

nbr_pre = compute_nbr(nir_pre, swir_pre)
nbr_post = compute_nbr(nir_post, swir_post)
dnbr = compute_dnbr(nbr_pre, nbr_post)

wildfire_result = detect_wildfire(dnbr)
print(f"Severity breakdown: {wildfire_result.severity_counts}")


Severity breakdown: {'no_data': 0, 'unburned': 1250, 'low': 0, 'moderate_low': 0, 'moderate_high': 0, 'high': 1250}


## 3. Generate the intelligence report

In [4]:
period_start = min(e.observation_time for e in events)
period_end = max(e.observation_time for e in events)

report = generate_intelligence_report(
    aoi=aoi,
    period_start=period_start,
    period_end=period_end,
    events=events,
    wildfire_result=wildfire_result,
    pixel_resolution_m=10.0,
)

print(f"AOI: {report.aoi_label}")
print(f"Period: {report.period_start} to {report.period_end}")
print(f"Total events: {report.total_events}")
print(f"Events by type: {report.events_by_type}")
print(f"Average confidence: {report.average_confidence}")
print(f"Highest-confidence event: {report.highest_confidence_event_id}")
print()
print(f"Burned area: {report.burned_area_hectares} hectares ({report.burned_area_percentage}% of scene)")
print(f"Severity breakdown: {report.severity_breakdown}")


AOI: Southern Africa (demo AOI)
Period: 2026-08-24T00:00:00+00:00 to 2026-08-24T05:00:00+00:00
Total events: 6
Events by type: {'wildfire': 6}
Average confidence: 0.6
Highest-confidence event: firms:demo:2

Burned area: 12.5 hectares (50.0% of scene)
Severity breakdown: {'no_data': 0, 'unburned': 1250, 'low': 0, 'moderate_low': 0, 'moderate_high': 0, 'high': 1250}


## 4. Limitations — generated dynamically, not boilerplate

In [5]:
for i, limitation in enumerate(report.limitations, 1):
    print(f"{i}. {limitation}\n")


1. Event data in this report comes from: NASA FIRMS (VIIRS_NOAA20_NRT). These are near-real-time satellite detections, not an exhaustive record — detection depends on satellite overpass timing, cloud cover, and fire size/intensity relative to sensor resolution.

2. All events in this report carry evidence level(s): observed. None are independently CONFIRMED. See this project's Responsible Use principles before treating any figure here as a verified fact.

3. Burn-severity classification uses dNBR thresholds (unburned<=0.1, low<=0.27, moderate_low<=0.44, moderate_high<=0.66, high>0.66) — an adaptation of USGS/FIREMON conventions for unscaled reflectance, not a universally validated standard. Different thresholds would yield a different reported affected area.

4. This report covers only wildfire monitoring. GeoWatch does not yet implement vegetation-decline, land-disturbance, or mining-related detection — no claims are made about non-fire environmental change in this AOI.



## 5. Export

In [6]:
json_output = report.to_json()
print(f"JSON report: {len(json_output)} characters")
print(json_output[:500] + "...")


JSON report: 3983 characters
{
  "aoi_label": "Southern Africa (demo AOI)",
  "aoi_bbox": [
    10.0,
    -35.0,
    40.0,
    -10.0
  ],
  "period_start": "2026-08-24T00:00:00+00:00",
  "period_end": "2026-08-24T05:00:00+00:00",
  "generated_at": "2026-08-26T11:34:35.688833+00:00",
  "total_events": 6,
  "events_by_type": {
    "wildfire": 6
  },
  "events_by_evidence_level": {
    "observed": 6
  },
  "average_confidence": 0.6,
  "highest_confidence_event_id": "firms:demo:2",
  "burned_area_hectares": 12.5,
  "burned_area...


In [7]:
csv_output = report.to_csv()
print("CSV event table:")
print(csv_output)


CSV event table:
event_id,event_type,evidence_level,confidence,latitude,longitude,observed_at,source,summary
firms:demo:0,wildfire,observed,0.3,-18.5,16.2,2026-08-24T00:00:00+00:00,NASA FIRMS (VIIRS_NOAA20_NRT),"Observed: wildfire near (-18.500, 16.200)"
firms:demo:1,wildfire,observed,0.6,-18.4,16.25,2026-08-24T01:00:00+00:00,NASA FIRMS (VIIRS_NOAA20_NRT),"Observed: wildfire near (-18.400, 16.250)"
firms:demo:2,wildfire,observed,0.9,-18.3,16.3,2026-08-24T02:00:00+00:00,NASA FIRMS (VIIRS_NOAA20_NRT),"Observed: wildfire near (-18.300, 16.300)"
firms:demo:3,wildfire,observed,0.3,-18.2,16.349999999999998,2026-08-24T03:00:00+00:00,NASA FIRMS (VIIRS_NOAA20_NRT),"Observed: wildfire near (-18.200, 16.350)"
firms:demo:4,wildfire,observed,0.6,-18.1,16.4,2026-08-24T04:00:00+00:00,NASA FIRMS (VIIRS_NOAA20_NRT),"Observed: wildfire near (-18.100, 16.400)"
firms:demo:5,wildfire,observed,0.9,-18.0,16.45,2026-08-24T05:00:00+00:00,NASA FIRMS (VIIRS_NOAA20_NRT),"Observed: wildfire near (-18.000, 16.450)"

## Milestone 5 complete — original MVP scope closed

This closes every item from the original Milestone 0-5 MVP definition:
NumPy foundation, NDVI/NBR/dNBR, wildfire detection with configurable
severity, NASA FIRMS live ingestion, PostGIS persistence, an interactive
dashboard, and now an automated intelligence report. Everything past
this point — Sentinel-1/SAR, machine learning, time-series recovery
tracking, the AI explainer layer, event streaming — is roadmap, exactly
as scoped from the start.